# Market Values

## Objective

In this notebook I create World Cup player market value datasets by country, tournament, and position group. The main outputs are:

- GKvals: top 1 goalkeeper per country per World Cup.
- Defvals: top 5 defenders per country per World Cup.
- MFvals: top 4 midfielders per country per World Cup.
- FWvals: top 4 forwards per country per World Cup.

I also create player availability indices from these position groups using whether each selected player appears in the World Cup roster data.

In [2]:
library(tidyverse)
library(here)

Valuations <- read.csv(here("1.DataCleaning-R", "kaggle_data", "player_valuations.csv"))
WCplayersIds <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "Matching.rds"))
WCRosters <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "FullRoster.rds"))
PlayerData <- read.csv(here("1.DataCleaning-R", "kaggle_data", "players.csv"))
CountriesWC <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "ELOSScores.rds")) %>%
  select(team)

Valuations <- Valuations %>%
  mutate(
    date = as.Date(date),
    tournament_id = case_when(
      date >= as.Date("2009-06-01") & date < as.Date("2010-06-01") ~ "WC-2010",
      date >= as.Date("2013-06-01") & date < as.Date("2014-06-01") ~ "WC-2014",
      date >= as.Date("2017-06-01") & date < as.Date("2018-06-01") ~ "WC-2018",
      date >= as.Date("2021-11-20") & date < as.Date("2022-11-20") ~ "WC-2022",
      date >= as.Date("2025-06-11") & date < as.Date("2026-06-11") ~ "WC-2026",
      TRUE ~ NA_character_
    )
  ) %>%
    filter(!is.na(tournament_id)) %>%
    group_by(player_id, tournament_id) %>%
    summarise(market_value_in_eur = mean(market_value_in_eur))


Valuations %>%
    print(n=20)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /Users/eialnisman/Desktop/WC2026Forecast

`summarise()` has grouped output by 'player_id'. You can override using the
`.groups` argument.


# A tibble: 85,709 x 3
# Groups:   player_id [39,080]
   player_id tournament_id market_value_in_eur
       <int> <chr>                       <dbl>
 1        10 WC-2010                 11625000 
 2        10 WC-2014                  1500000 
 3        26 WC-2010                  4500000 
 4        26 WC-2014                  5000000 
 5        26 WC-2018                   875000 
 6        65 WC-2010                 31000000 
 7        65 WC-2014                  4250000 
 8        77 WC-2010                 16000000 
 9        77 WC-2014                   500000 
10        80 WC-2010                  1750000 
11        80 WC-2014                  1000000 
12        80 WC-2018                   133333.
13       109 WC-2010                  5625000 
14       109 WC-2014                   450000 
15       123 WC-2010                  3333333.
16       132 WC-2010                 12000000 
17       132 WC-2014                  2000000 
18       132 WC-2018                   350000 
19    

## Player Countries and World Cup Rosters

In [3]:
PlayerCountries <- PlayerData %>%
    select(player_id, country_of_citizenship) 

ValuationsCountry <- left_join(Valuations, PlayerCountries, by= "player_id")

ValuationsWC <- ValuationsCountry %>% 
    filter(country_of_citizenship %in% CountriesWC$team)

ValuationsWC %>%
    print(n=20)

# A tibble: 65,969 x 4
# Groups:   player_id [30,000]
   player_id tournament_id market_value_in_eur country_of_citizenship
       <int> <chr>                       <dbl> <chr>                 
 1        10 WC-2010                 11625000  Germany               
 2        10 WC-2014                  1500000  Germany               
 3        26 WC-2010                  4500000  Germany               
 4        26 WC-2014                  5000000  Germany               
 5        26 WC-2018                   875000  Germany               
 6        77 WC-2010                 16000000  Brazil                
 7        77 WC-2014                   500000  Brazil                
 8        80 WC-2010                  1750000  Germany               
 9        80 WC-2014                  1000000  Germany               
10        80 WC-2018                   133333. Germany               
11       109 WC-2010                  5625000  Brazil                
12       109 WC-2014                

In [4]:
ValuationsWC <- left_join(ValuationsWC, WCplayersIds, by=c("player_id"="player_id_kag"))

In [5]:
ValuationsWC <- ValuationsWC %>%
    mutate(
        inWC = ifelse(
            paste(player_id_fj, tournament_id) %in%
                paste(WCRosters$player_id, WCRosters$tournament_id),
            1,
            0
        )
    )

sum(ValuationsWC$inWC)

[1] 2111

## Matching Checks

Since some Kaggle players do not match to Fjelstul IDs, I inspect the highest market value unmatched players. These are the highest leverage cases for manual review.

In [6]:
playernamesidkag <- PlayerData %>%
    select(player_id, name)

left_join(ValuationsWC, playernamesidkag, by = "player_id") %>%
    filter(
        is.na(player_id_fj),
        tournament_id != "WC-2026",
        country_of_citizenship %in% WCRosters$team_name
    ) %>%
    group_by(tournament_id) %>%
    slice_max(market_value_in_eur, n = 20, with_ties = FALSE) %>%
    arrange(tournament_id, desc(market_value_in_eur)) %>%
    select(
        tournament_id,
        name,
        player_id,
        market_value_in_eur,
    ) %>%
    print(n = Inf, width = Inf)


# A tibble: 80 x 4
# Groups:   tournament_id [4]
   tournament_id name                        player_id market_value_in_eur
   <chr>         <chr>                           <dbl>               <dbl>
 1 WC-2010       "Zlatan Ibrahimovi\u0107"        3455           45666667.
 2 WC-2010       "Rio Ferdinand"                  3235           33000000 
 3 WC-2010       "Alexandre Pato"                37579           31250000 
 4 WC-2010       "Esteban Cambiasso"              7520           29000000 
 5 WC-2010       "Diego"                          4248           28000000 
 6 WC-2010       "Andrey Arshavin"               15378           26000000 
 7 WC-2010       "Giuseppe Rossi"                19104           23000000 
 8 WC-2010       "Lassana Diarra"                23914           22000000 
 9 WC-2010       "Anderson"                      31645           21500000 
10 WC-2010       "Bosingwa"                       9813           20750000 
11 WC-2010       "Adriano"                       34

In [7]:
ValuationsWC %>%
    print(n=10)

# A tibble: 65,969 x 7
# Groups:   player_id [30,000]
   player_id tournament_id market_value_in_eur country_of_citizenship full_name 
       <dbl> <chr>                       <dbl> <chr>                  <chr>     
 1        10 WC-2010                 11625000  Germany                Miroslav ~
 2        10 WC-2014                  1500000  Germany                Miroslav ~
 3        26 WC-2010                  4500000  Germany                Roman Wei~
 4        26 WC-2014                  5000000  Germany                Roman Wei~
 5        26 WC-2018                   875000  Germany                Roman Wei~
 6        77 WC-2010                 16000000  Brazil                 Lucio     
 7        77 WC-2014                   500000  Brazil                 Lucio     
 8        80 WC-2010                  1750000  Germany                NA        
 9        80 WC-2014                  1000000  Germany                NA        
10        80 WC-2018                   133333. Germany 

## Position Value Datasets

The next step creates the four position-specific market value datasets. Each dataset keeps players whose citizenship country appears in that World Cup and then selects the highest value players within each country and tournament.

In [8]:
player_positions <- PlayerData %>%
  select(player_id, position) %>%
  distinct(player_id, .keep_all = TRUE)

wc_teams <- WCRosters %>%
  distinct(tournament_id, team_name)

final_val_cols <- c(
  "full_name",
  "tournament_id",
  "country_of_citizenship",
  "player_id_kag",
  "player_id_fj",
  "market_value_in_eur",
  "inWC"
)

BaseVals <- ValuationsWC %>%
  select(
    player_id,
    tournament_id,
    market_value_in_eur,
    country_of_citizenship,
    full_name,
    player_id_fj,
    inWC
  ) %>%
  left_join(player_positions, by = "player_id") %>%
  semi_join(
    wc_teams,
    by = c(
      "tournament_id" = "tournament_id",
      "country_of_citizenship" = "team_name"
    )
  )

top_position_vals <- function(position_name, n_players) {
  BaseVals %>%
    filter(position == position_name) %>%
    group_by(tournament_id, country_of_citizenship) %>%
    slice_max(market_value_in_eur, n = n_players, with_ties = FALSE) %>%
    ungroup() %>%
    rename(player_id_kag = player_id) %>%
    select(all_of(final_val_cols))
}

GKvals <- top_position_vals("Goalkeeper", 1)
Defvals <- top_position_vals("Defender", 5)
MFvals <- top_position_vals("Midfield", 4)
FWvals <- top_position_vals("Attack", 4)

print(names(BaseVals))
print(table(BaseVals$position, useNA = "ifany"))
print(list(
  GKvals = dim(GKvals),
  Defvals = dim(Defvals),
  MFvals = dim(MFvals),
  FWvals = dim(FWvals)
))

[1] "player_id"              "tournament_id"          "market_value_in_eur"   
[4] "country_of_citizenship" "full_name"              "player_id_fj"          
[7] "inWC"                   "position"              

    Attack   Defender Goalkeeper   Midfield    Missing 
      9384      11964       4158      10385         63 
$GKvals
[1] 116   7

$Defvals
[1] 587   7

$MFvals
[1] 469   7

$FWvals
[1] 471   7



I inspect a sample of the created position datasets before building the indices.

In [9]:
head(MFvals)

GKvals %>%
    filter(country_of_citizenship=="Argentina")

full_name,tournament_id,country_of_citizenship,player_id_kag,player_id_fj,market_value_in_eur,inWC
<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
Hassan Yebda,WC-2010,Algeria,12351,P-70560,4.9e+06,1
Mehdi Lacen,WC-2010,Algeria,35539,P-23484,3.2e+06,1
Sofiane Feghouli,WC-2010,Algeria,57162,P-12165,2.9e+06,0
Ryad Boudebouz,WC-2010,Algeria,77826,P-70903,1.8e+06,1
NA,WC-2010,Argentina,7520,NA,2.9e+07,0
Javier Mascherano,WC-2010,Argentina,19981,P-77063,2.6e+07,1


full_name,tournament_id,country_of_citizenship,player_id_kag,player_id_fj,market_value_in_eur,inWC
<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
NA,WC-2010,Argentina,7669,NA,6000000,0
Willy Caballero,WC-2014,Argentina,19948,P-17853,5000000,0
Geronimo Rulli,WC-2018,Argentina,229604,P-36188,11333333,0
Emiliano Martinez,WC-2022,Argentina,111873,P-13162,28333333,1


## Player Availability Index

The index compares the total value of the selected top players with the value of the selected players who were actually in the World Cup roster. This is an imperfect but useful proxy for injuries, suspensions, selection decisions, and other absences.

In [10]:
GKind <- GKvals %>%
    group_by( tournament_id, country_of_citizenship) %>%
        summarise(inWC = sum(inWC), value = sum(market_value_in_eur))

head(GKind)

`summarise()` has grouped output by 'tournament_id'. You can override using the
`.groups` argument.


tournament_id,country_of_citizenship,inWC,value
<chr>,<chr>,<dbl>,<dbl>
WC-2010,Algeria,1,600000
WC-2010,Argentina,0,6000000
WC-2010,Australia,1,4000000
WC-2010,Brazil,1,24000000
WC-2010,Cameroon,1,9666667
WC-2010,Chile,1,3333333


In [11]:
Defind <- Defvals %>%
    group_by(tournament_id, country_of_citizenship) %>%
    summarise(
        maxvalue = sum(market_value_in_eur, na.rm = TRUE),
        value = sum(market_value_in_eur[inWC == 1], na.rm = TRUE),
        .groups = "drop"
    ) %>%
    mutate(index = value / maxvalue)

head(Defind)

tournament_id,country_of_citizenship,maxvalue,value,index
<chr>,<chr>,<dbl>,<dbl>,<dbl>
WC-2010,Algeria,7212500,5500000,0.7625650
WC-2010,Argentina,57100000,26833333,0.4699358
WC-2010,Australia,5075000,4250000,0.8374384
WC-2010,Brazil,109750000,89750000,0.8177677
WC-2010,Cameroon,22400000,12750000,0.5691964
WC-2010,Chile,14416667,11250000,0.7803468


In [12]:
MFind <- MFvals %>%
    group_by(tournament_id, country_of_citizenship) %>%
    summarise(
        maxvalue = sum(market_value_in_eur, na.rm = TRUE),
        value = sum(market_value_in_eur[inWC == 1], na.rm = TRUE),
        .groups = "drop"
    ) %>%
    mutate(index = value / maxvalue)

head(MFind)

tournament_id,country_of_citizenship,maxvalue,value,index
<chr>,<chr>,<dbl>,<dbl>,<dbl>
WC-2010,Algeria,12800000,9900000,0.7734375
WC-2010,Argentina,91666667,26000000,0.2836364
WC-2010,Australia,6725000,3825000,0.5687732
WC-2010,Brazil,124166667,74666667,0.6013423
WC-2010,Cameroon,40166667,40166667,1.0000000
WC-2010,Chile,30750000,9000000,0.2926829


In [13]:
FWind <- FWvals %>%
    group_by(tournament_id, country_of_citizenship) %>%
    summarise(
        maxvalue = sum(market_value_in_eur, na.rm = TRUE),
        value = sum(market_value_in_eur[inWC == 1], na.rm = TRUE),
        .groups = "drop"
    ) %>%
    mutate(index = value / maxvalue)

head(FWind)

tournament_id,country_of_citizenship,maxvalue,value,index
<chr>,<chr>,<dbl>,<dbl>,<dbl>
WC-2010,Algeria,9375000,7375000,0.78666667
WC-2010,Argentina,166000000,166000000,1.00000000
WC-2010,Australia,7050000,475000,0.06737589
WC-2010,Brazil,87750000,31500000,0.35897436
WC-2010,Cameroon,47500000,40900000,0.86105263
WC-2010,Chile,20766667,18916667,0.91091493


## Save Outputs

In [14]:
saveRDS(GKind, here("1.DataCleaning-R", "Data", "RDS", "GKind.rds"))
saveRDS(Defind, here("1.DataCleaning-R", "Data", "RDS", "Defind.rds"))
saveRDS(MFind, here("1.DataCleaning-R", "Data", "RDS", "MFind.rds"))
saveRDS(FWind, here("1.DataCleaning-R", "Data", "RDS", "FWind.rds"))